In [5]:
from catalyst.debug.compiler_functions import get_compilation_stage
import pennylane as qml
from catalyst.third_party.oqd import OQDDevice

import os
import shutil
import pathlib
import numpy as np

from functools import partial

########################################################################################

for f in os.listdir():
    if f.startswith("oqd_circuit_ode") and os.path.isdir(f):
        shutil.rmtree(pathlib.Path(f))

compile_results = pathlib.Path("oqd_circuit_ode")
openapl_file_name = "oqd_circuit_ode.openapl.json"


toml_files = {
    "device-toml-loc": "/home/user/oqd-catalyst/scripts/calibration_data/device.toml",
    "qubit-toml-loc": "/home/user/oqd-catalyst/scripts/calibration_data/qubit.toml",
    "gate-to-pulse-toml-loc": "/home/user/oqd-catalyst/scripts/calibration_data/gate.toml",
}

toml_files = " ".join([f"{k}={v}" for k, v in toml_files.items()])


OQD_PIPELINES = [
    (
        "DeviceAgnosticPipeline",
        [
            "quantum-compilation-stage",
            "hlo-lowering-stage",
            "gradient-lowering-stage",
            "bufferization-stage",
        ],
    ),
    (
        "IonDecompositionStage",
        [
            "func.func(ions-decomposition)",
            "func.func(merge-rotations)",
            "func.func(prune-zero-rotations)",
        ],
    ),
    (
        "IonDialectLoweringStage",
        [
            f"func.func(gates-to-pulses{{{toml_files}}})",
        ],
    ),
    ("IonToLLVMDialectConversion", ["convert-ion-to-llvm"]),
    ("MLIRToLLVMDialectConversion", ["llvm-dialect-lowering-stage"]),
]

ALT_PIPELINES = [
    (
        "QuantumCompilationStage",
        [
            "split-multiple-tapes",
            "builtin.module(apply-transform-sequence)",
            "inline-nested-module",
            "lower-mitigation",
            "adjoint-lowering",
        ],
    ),
    (
        "HLOLoweringStage",
        [
            "canonicalize",
            "func.func(chlo-legalize-to-stablehlo)",
            "func.func(stablehlo-legalize-control-flow)",
            "func.func(stablehlo-aggressive-simplification)",
            "stablehlo-legalize-to-linalg",
            "func.func(stablehlo-legalize-to-std)",
            "func.func(stablehlo-legalize-sort)",
            "stablehlo-convert-to-signless",
            "canonicalize",
            "scatter-lowering",
            "hlo-custom-call-lowering",
            "cse",
            "func.func(linalg-detensorize{aggressive-mode})",
            "detensorize-scf",
            "detensorize-function-boundary",
            "canonicalize",
            "symbol-dce",
        ],
    ),
    (
        "IonDecompositionStage",
        [
            "ions-decomposition",
            "merge-rotations",
            "prune-zero-rotations",
            "merge-rotations",
        ],
    ),
    (
        "GradientLoweringStage",
        [
            "annotate-invalid-gradient-functions",
            "lower-gradients",
        ],
    ),
    (
        "BufferizationStage",
        [
            "inline",
            "convert-tensor-to-linalg",
            "convert-elementwise-to-linalg",
            "gradient-preprocess",
            "one-shot-bufferize{bufferize-function-boundaries         allow-return-allocs-from-loops         function-boundary-type-conversion=identity-layout-map         unknown-type-conversion=identity-layout-map}",
            "canonicalize",
            "gradient-postprocess",
            "func.func(buffer-hoisting)",
            "func.func(buffer-loop-hoisting)",
            "func.func(buffer-deallocation)",
            "convert-arraylist-to-memref",
            "convert-bufferization-to-memref",
            "canonicalize",
            "cp-global-memref",
        ],
    ),
    (
        "MLIRToLLVMDialectConversion",
        [
            "expand-realloc",
            "convert-gradient-to-llvm",
            "memrefcpy-to-linalgcpy",
            "func.func(convert-linalg-to-loops)",
            "convert-scf-to-cf",
            "expand-strided-metadata",
            "lower-affine",
            "arith-expand",
            "convert-complex-to-standard",
            "convert-complex-to-llvm",
            "convert-math-to-llvm",
            "convert-math-to-libm",
            "convert-arith-to-llvm",
            "memref-to-llvm-tbaa",
            "finalize-memref-to-llvm{use-generic-functions}",
            "convert-index-to-llvm",
            "convert-catalyst-to-llvm",
            "convert-quantum-to-llvm",
            "emit-catalyst-py-interface",
            "canonicalize",
            "reconcile-unrealized-casts",
            "gep-inbounds",
            "register-inactive-callback",
        ],
    ),
]


oqd_dev = OQDDevice(
    backend="default",
    wires=26,
    openapl_file_name=(compile_results / openapl_file_name).as_posix(),
)


with open("lotka_volterra_term_1.qasm", "r") as f:
    qasm_string = f.read()


@qml.set_shots(1)
@qml.qnode(oqd_dev)
def oqd_circuit_ode():
    qml.from_qasm(qasm_string)()
    return qml.counts(wires=0)


QJIT_CIRCUIT = qml.qjit(
    oqd_circuit_ode, pipelines=OQD_PIPELINES, keep_intermediate=True, verbose=True
)

ALT_QJIT_CIRCUIT = qml.qjit(oqd_circuit_ode, pipelines=ALT_PIPELINES)

print("{:=^100}".format("\033[1;32m Compiled circuit \033[0m"))
print(get_compilation_stage(QJIT_CIRCUIT, stage="IonDialectLoweringStage"))


[LIB] Running compiler driver in /home/user/oqd-catalyst/scripts/oqd_circuit_ode
[SYSTEM] /home/user/oqd-catalyst/frontend/catalyst/utils/../../../mlir/build/bin/catalyst -o /home/user/oqd-catalyst/scripts/oqd_circuit_ode/oqd_circuit_ode.ll --module-name oqd_circuit_ode --workspace /home/user/oqd-catalyst/scripts/oqd_circuit_ode -verify-each=false --catalyst-pipeline DeviceAgnosticPipeline(quantum-compilation-stage;hlo-lowering-stage;gradient-lowering-stage;bufferization-stage),IonDecompositionStage(func.func(ions-decomposition);func.func(merge-rotations);func.func(prune-zero-rotations)),IonDialectLoweringStage(func.func(gates-to-pulses{device-toml-loc=/home/user/oqd-catalyst/scripts/calibration_data/device.toml qubit-toml-loc=/home/user/oqd-catalyst/scripts/calibration_data/qubit.toml gate-to-pulse-toml-loc=/home/user/oqd-catalyst/scripts/calibration_data/gate.toml})),IonToLLVMDialectConversion(convert-ion-to-llvm),MLIRToLLVMDialectConversion(llvm-dialect-lowering-stage), --keep-inter

In [6]:
import json

print(qml.draw(oqd_circuit_ode)())

with open(compile_results / "oqd_circuit_ode.draw.txt", "w") as f:
    f.write(qml.draw(oqd_circuit_ode)())

with open(compile_results / "oqd_circuit_ode.catalyst.specs", "w") as f:
    f.write(qml.specs(ALT_QJIT_CIRCUIT)().__str__())

QJIT_CIRCUIT()

print(json.dumps(json.load(open(compile_results / openapl_file_name)), indent=2))

0: ──X────────╭●────────────╭●──X────────╭●────────────╭●─┤  Counts
1: ──RY(0.79)─╰X──RY(-0.79)─╰X──RY(2.36)─╰X──RY(-2.36)─╰X─┤        
{
  "class_": "AtomicCircuit",
  "protocol": {
    "class_": "SequentialProtocol",
    "sequence": [
      {
        "class_": "ParallelProtocol",
        "sequence": [
          {
            "beam": {
              "class_": "Beam",
              "detuning": {
                "class_": "MathNum",
                "value": 208570336271826.38
              },
              "phase": {
                "class_": "MathNum",
                "value": 0.0
              },
              "polarization": [
                1,
                0,
                0
              ],
              "rabi": {
                "class_": "MathNum",
                "value": 6283185307.179586
              },
              "target": 0,
              "transition": {
                "class_": "Transition",
                "einsteinA": 41050903.119868636,
                "label"

In [7]:
from oqd_core.interface.atomic import AtomicCircuit
from oqd_core.compiler.atomic.canonicalize import canonicalize_atomic_circuit_factory
from oqd_compiler_infrastructure import Chain, Post
from oqd_bare_metal.compiler.codegen import AtomicToTestbenchV2
from oqd_bare_metal.compiler.optim import (
    SpectrumCoreRemapping,
    SpectrumPrune,
    SpectrumUnwrapResets,
)
import ast_comments as ast


circuit = AtomicCircuit.model_validate_json(
    json.dumps(json.load(open(compile_results / openapl_file_name)), indent=2)
)


compiler = Chain(
    canonicalize_atomic_circuit_factory(),
    Post(
        AtomicToTestbenchV2(
            device_params="./calibration_data/testbench_params.toml",
        ),
    ),
)
optimization_pass = Chain(
    Post(SpectrumCoreRemapping()),
    Post(SpectrumUnwrapResets()),
    Post(SpectrumPrune()),
)

unopt_artiq_experiment = compiler(circuit)
artiq_experiment = optimization_pass(unopt_artiq_experiment)

print(ast.unparse(ast.fix_missing_locations(artiq_experiment)))

with open(compile_results / "oqd_circuit_ode.artiq.py", "w") as f:
    f.write(ast.unparse(ast.fix_missing_locations(artiq_experiment)))

import numpy as np
from artiq.experiment import *

class TestbenchV2Experiment(EnvExperiment):

    @rpc(flags={'async'})
    def transfer_data(self, key, index, value):
        self.mutate_dataset(key=key, index=index, value=value)

    def build(self):
        self.setattr_device('core')
        self.setattr_device('awg')
        self.setattr_device('ttl0')
        self.setattr_device('ttl4')
        self.setattr_device('ttl6')
        self.setattr_device('ttl7')
        self.setattr_device('urukul0_ch0')
        self.setattr_device('urukul0_ch1')
        self.setattr_device('urukul0_ch2')
        self.setattr_device('urukul0_ch3')
        self.setattr_device('urukul1_ch0')
        self.setattr_device('urukul1_ch1')
        self.setattr_device('urukul1_ch2')
        self.setattr_device('urukul1_ch3')

    def program_awg(self):
        # init Spectrum AWG
        self.awg.start()
        self.awg.card_mode('dds')
        self.awg.channel_enable_out(True)
        self.awg.channels_out

In [8]:
# from oqd_core.interface.atomic import AtomicCircuit
# from oqd_core.compiler.atomic.canonicalize import canonicalize_atomic_circuit_factory
# from oqd_compiler_infrastructure import Chain, Post
# from oqd_bare_metal.compiler.codegen import AtomicToBloodstoneV1
# from oqd_bare_metal.compiler.optim import (
#     SpectrumCoreRemapping,
#     SpectrumPrune,
#     SpectrumUnwrapResets,
# )
# import ast_comments as ast


# circuit = AtomicCircuit.model_validate_json(
#     json.dumps(json.load(open(compile_results / openapl_file_name)), indent=2)
# )


# compiler = Chain(
#     canonicalize_atomic_circuit_factory(),
#     Post(
#         AtomicToBloodstoneV1(device_params="./calibration_data/bloodstone_params.toml")
#     ),
# )
# optimization_pass = Chain(
#     Post(SpectrumCoreRemapping(device="raman_awg")),
#     Post(SpectrumUnwrapResets()),
#     Post(SpectrumPrune()),
# )

# unopt_artiq_experiment = compiler(circuit)
# artiq_experiment = optimization_pass(unopt_artiq_experiment)


# with open(compile_results / "oqd_circuit.artiq.py", "w") as f:
#     f.write(ast.unparse(ast.fix_missing_locations(artiq_experiment)))